# Face Recognition — Siamese Networks & Triplet Loss

> Based on Stanford CS230 — C4M4, Andrew Ng / DeepLearning.AI

Face recognition must scale to a gallery of thousands of identities — even identities not seen at training time. The key insight: learn an **embedding** $f(x) \in \mathbb{R}^{128}$ where same-person pairs are close and different-person pairs are far apart.

## Learning Objectives

1. Distinguish face **verification** (1:1) from face **recognition** (1:K)
2. Explain one-shot learning with Siamese networks
3. Define and compute the triplet loss
4. Visualise embedding space for identity clusters

## Triplet Loss

Given an anchor $A$, positive $P$ (same person), and negative $N$ (different person):

$$\mathcal{L}(A, P, N) = \max\!\bigl(\underbrace{\|f(A)-f(P)\|^2}_{\text{pull together}} - \underbrace{\|f(A)-f(N)\|^2}_{\text{push apart}} + \alpha,\; 0\bigr)$$

The margin $\alpha$ prevents the network from collapsing to $f(x) = \mathbf{0}$ for all $x$.

## Siamese Network

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 200" width="680" height="200" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="ta" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Input images -->
  <rect x="10"  y="60"  width="80" height="80" rx="5" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.6"/>
  <text x="50"  y="96"  text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">Anchor (A)</text>
  <text x="50"  y="112" text-anchor="middle" fill="#3d7a5e" font-size="9">same person</text>
  <rect x="10"  y="10"  width="80" height="40" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="50"  y="33"  text-anchor="middle" fill="#3a7bbf" font-size="10" font-weight="bold">Positive (P)</text>
  <rect x="10"  y="150" width="80" height="40" rx="5" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.5"/>
  <text x="50"  y="173" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">Negative (N)</text>
  <!-- Shared network blocks -->
  <rect x="160" y="10"  width="100" height="40" rx="5" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.5"/>
  <text x="210" y="33"  text-anchor="middle" fill="#a07010" font-size="10">CNN f(P)</text>
  <rect x="160" y="60"  width="100" height="80" rx="5" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.5"/>
  <text x="210" y="96"  text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">Shared CNN</text>
  <text x="210" y="112" text-anchor="middle" fill="#a07010" font-size="9">f(A)</text>
  <rect x="160" y="150" width="100" height="40" rx="5" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.5"/>
  <text x="210" y="173" text-anchor="middle" fill="#a07010" font-size="10">CNN f(N)</text>
  <!-- Embedding outputs -->
  <rect x="340" y="10"  width="100" height="40" rx="5" fill="#56b6c2" fill-opacity="0.15" stroke="#56b6c2" stroke-width="1.5"/>
  <text x="390" y="33"  text-anchor="middle" fill="#2a7080" font-size="10">embed P ∈ ℝ¹²⁸</text>
  <rect x="340" y="60"  width="100" height="80" rx="5" fill="#56b6c2" fill-opacity="0.15" stroke="#56b6c2" stroke-width="1.5"/>
  <text x="390" y="96"  text-anchor="middle" fill="#2a7080" font-size="10" font-weight="bold">embed A ∈ ℝ¹²⁸</text>
  <rect x="340" y="150" width="100" height="40" rx="5" fill="#56b6c2" fill-opacity="0.15" stroke="#56b6c2" stroke-width="1.5"/>
  <text x="390" y="173" text-anchor="middle" fill="#2a7080" font-size="10">embed N ∈ ℝ¹²⁸</text>
  <!-- Loss box -->
  <rect x="500" y="70" width="160" height="60" rx="6" fill="#c678dd" fill-opacity="0.12" stroke="#c678dd" stroke-width="1.6"/>
  <text x="580" y="96"  text-anchor="middle" fill="#8a3ab8" font-size="10" font-weight="bold">Triplet Loss</text>
  <text x="580" y="113" text-anchor="middle" fill="#8a3ab8" font-size="9">max(d(A,P)−d(A,N)+α, 0)</text>
  <!-- Arrows -->
  <line x1="90" y1="100" x2="160" y2="100" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="90" y1="30"  x2="160" y2="30"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="90" y1="170" x2="160" y2="170" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="260" y1="100" x2="340" y2="100" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="260" y1="30"  x2="340" y2="30"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="260" y1="170" x2="340" y2="170" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="440" y1="100" x2="500" y2="100" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="440" y1="30"  x2="470" y2="30"  stroke="#aaa" stroke-width="1.2"/>
  <line x1="470" y1="30"  x2="470" y2="80"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
  <line x1="440" y1="170" x2="470" y2="170" stroke="#aaa" stroke-width="1.2"/>
  <line x1="470" y1="170" x2="470" y2="120" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta)"/>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def l2(a, b): return np.sum((a - b)**2)

def triplet_loss(anchor, pos, neg, alpha=0.2):
    return max(l2(anchor, pos) - l2(anchor, neg) + alpha, 0)

# ---- Simulate embeddings for 5 identities ----
np.random.seed(42)
n_ids, n_samples = 5, 6
embed_dim = 2  # 2D for visualisation

# Cluster centres
centres = np.array([[1,1],[3,1],[2,3],[0,3],[4,3]], dtype=float)
labels  = []
embeds  = []
for i, c in enumerate(centres):
    for _ in range(n_samples):
        embeds.append(c + 0.3 * np.random.randn(2))
        labels.append(i)
embeds = np.array(embeds)
labels = np.array(labels)

# ---- Triplet loss for one triplet ----
anchor = embeds[0]
positive = embeds[1]                   # same identity, different sample
negative = embeds[n_samples]           # different identity

loss_val = triplet_loss(anchor, positive, negative, alpha=0.2)
print("Triplet Loss Demonstration")
print(f"  d(A,P) = {l2(anchor,positive):.4f}   d(A,N) = {l2(anchor,negative):.4f}")
print(f"  loss   = max({l2(anchor,positive):.4f} - {l2(anchor,negative):.4f} + 0.2, 0)")
print(f"         = {loss_val:.4f}")

# ---- Vary margin to see loss landscape ----
alphas = np.linspace(0, 2, 200)
losses = [max(l2(anchor, positive) - l2(anchor, negative) + a, 0) for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Siamese Network — Triplet Loss & Embedding Space', fontsize=13, fontweight='bold')

axes[0].plot(alphas, losses, color=P[0], lw=2)
axes[0].axvline(0.2, color=P[2], ls='--', lw=1.5, label='α = 0.2 (used)')
axes[0].set_xlabel('Margin α')
axes[0].set_ylabel('Triplet Loss')
axes[0].set_title('Triplet Loss vs Margin α')
axes[0].legend()
axes[0].grid(True)

# ---- Embedding scatter ----
id_names  = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
id_colors = [P[0], P[1], P[2], P[3], P[4]]
ax = axes[1]
for i, (name, color) in enumerate(zip(id_names, id_colors)):
    mask = labels == i
    ax.scatter(embeds[mask, 0], embeds[mask, 1], color=color, s=60,
               alpha=0.8, label=name, zorder=3)
    ax.scatter(*centres[i], marker='*', s=200, color=color, edgecolors='black', lw=0.5, zorder=5)

ax.set_title('2D Embedding Space (identity clusters)')
ax.set_xlabel('embedding dim 1')
ax.set_ylabel('embedding dim 2')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True)

# Show triplet
ax.annotate('', xy=embeds[n_samples], xytext=embeds[0],
            arrowprops=dict(arrowstyle='->', color=P[1], lw=2))
ax.annotate('', xy=embeds[1], xytext=embeds[0],
            arrowprops=dict(arrowstyle='->', color=P[3], lw=2))
ax.text(*embeds[0] + [-0.15, 0.05], 'A', color='black', fontsize=9, fontweight='bold')
ax.text(*embeds[1] + [0.05, 0.05],  'P', color=P[3],   fontsize=9, fontweight='bold')
ax.text(*embeds[n_samples] + [0.05, 0.05], 'N', color=P[1], fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()
